# 소재정보학 실습: 딥러닝 알고리즘 기초

강의 「딥러닝 알고리즘 기초」의 내용을 직접 코드로 확인하는 실습입니다.

| Part | 내용 | 강의 슬라이드 | 예상 시간 |
|---|---|---|---|
| 1 | 퍼셉트론과 XOR | Perceptron, Multi-layer perceptron | 10분 |
| 2 | 순전파와 역전파를 손으로 계산하기 | Learning process, 계산 예 | 15분 |
| 3 | PyTorch로 와인 분류 | Example of ANN, Gradient descent, Optimizer | 20분 |
| 4 | 과적합과 regularization | Overfitting과 일반화, Regularization | 10분 |
| 5 | 결정 그래프 신경망(CGCNN) 직접 만들기 | 구조의 그래프 표현 – CGCNN convolution | 30분 |
| 6 | 데이터 분할과 누출 | 소재 데이터의 주의점 | 10분 |
| 7 | (선택) 사전학습된 범용 퍼텐셜 CHGNet 사용 | M3GNet, CHGNet | 15분 |

**준비**

```bash
pip install numpy matplotlib scikit-learn torch
pip install chgnet ase        # Part 7(선택)에만 필요
```

- GPU는 필요 없습니다. 전체 실행은 노트북 CPU에서 몇 분이면 끝납니다.
- 셀은 위에서부터 순서대로 실행하세요.
- `연습` 표시가 있는 곳은 값을 바꿔 가며 결과가 어떻게 달라지는지 직접 확인해 보세요.

In [1]:
!pip install matplotlib
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
print("torch", torch.__version__, "| numpy", np.__version__)

OSError: [WinError 4551] 애플리케이션 제어 정책에서 이 파일을 차단했습니다. Error loading "C:\Users\yeale\miniconda3\envs\class\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

---
## Part 1. 퍼셉트론과 XOR

퍼셉트론 하나는 입력의 가중합을 구한 뒤 계단 함수로 0 또는 1을 출력합니다.

$$y = \text{step}(w_1 x_1 + w_2 x_2 + b)$$

강의에서 본 것처럼 AND는 $x_1 + x_2 - 1.5$, OR는 $x_1 + x_2 - 0.5$로 만들 수 있습니다.

In [ ]:
def perceptron(x, w, b):
    # 가중합이 0보다 크면 1, 아니면 0
    return (x @ w + b > 0).astype(int)

X = np.array([[0, 0], [1, 0], [0, 1], [1, 1]])

AND  = lambda x: perceptron(x, np.array([ 1.0,  1.0]), -1.5)
OR   = lambda x: perceptron(x, np.array([ 1.0,  1.0]), -0.5)
NAND = lambda x: perceptron(x, np.array([-1.0, -1.0]),  1.5)

print("x1 x2 | AND OR NAND")
for x, a, o, n in zip(X, AND(X), OR(X), NAND(X)):
    print(f" {x[0]}  {x[1]} |  {a}   {o}   {n}")

XOR는 직선 하나로 나눌 수 없습니다. 하지만 NAND와 OR의 출력을 다시 AND에 넣으면(= 은닉층을 하나 두면) 만들 수 있습니다.

In [ ]:
def XOR(x):
    hidden = np.stack([NAND(x), OR(x)], axis=1)   # 은닉층: 퍼셉트론 2개
    return AND(hidden)                            # 출력층: 퍼셉트론 1개

print("x1 x2 | XOR")
for x, y in zip(X, XOR(X)):
    print(f" {x[0]}  {x[1]} |  {y}")

# 결정 경계 그리기
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
gx, gy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
grid = np.stack([gx.ravel(), gy.ravel()], axis=1)
for ax, (name, f) in zip(axes, [("AND", AND), ("OR", OR), ("XOR (2 layers)", XOR)]):
    ax.contourf(gx, gy, f(grid).reshape(gx.shape), levels=[-0.5, 0.5, 1.5], colors=["#dce6f2", "#f8cbad"])
    ax.scatter(X[:, 0], X[:, 1], c=f(X), cmap="coolwarm", edgecolors="k", s=120, zorder=3)
    ax.set_title(name); ax.set_xlabel("x1"); ax.set_ylabel("x2")
plt.tight_layout(); plt.show()

**연습 1.** 가중치와 편향만 바꿔서 NOR(둘 다 0일 때만 1) 퍼셉트론을 만들어 보세요.

---
## Part 2. 순전파와 역전파를 손으로 계산하기

강의 슬라이드의 작은 신경망(입력 2, 은닉 2, 출력 2)을 그대로 옮깁니다.

- 입력 $x = (1, 2)$, 정답 $t = (1, 0)$ (설명을 위해 가정한 값)
- 활성화 함수는 sigmoid, 비용함수는 $C = \tfrac{1}{2}\sum_k (y_k - t_k)^2$

가중치 행렬은 `W[i, j]` = "i번 노드에서 j번 노드로 가는 가중치"로 둡니다. 예를 들어 `W1[0, 1]`은 슬라이드의 $w_{12}$입니다.

In [ ]:
sigmoid = lambda z: 1.0 / (1.0 + np.exp(-z))

x  = np.array([1.0, 2.0])
t  = np.array([1.0, 0.0])
W1 = np.array([[0.1, 0.2],     # w11, w12
               [0.3, 0.4]])    # w21, w22
b1 = np.array([0.5, 0.5])      # 은닉 노드마다 편향 하나씩 (초기값은 같음)
W2 = np.array([[0.7, 0.8],     # w31, w32
               [0.9, 0.1]])    # w41, w42
b2 = np.array([0.3, 0.3])

# ----- 순전파 -----
g = W1.T @ x + b1      # 은닉층 가중합
h = sigmoid(g)         # 은닉층 출력
z = W2.T @ h + b2      # 출력층 가중합
y = sigmoid(z)         # 최종 출력
cost = 0.5 * np.sum((y - t) ** 2)

print("g =", g.round(3), " h =", h.round(3))
print("z =", z.round(3), " y =", y.round(3))
print(f"C = {cost:.3f}")

슬라이드의 값(h = 0.769, 0.818 / y = 0.828, 0.730 / C = 0.281)과 같은지 확인하세요.

이제 역전파입니다. 연쇄 법칙을 출력층부터 거꾸로 적용합니다.

- 출력층 오차 신호: $\delta_k = (y_k - t_k)\, y_k (1 - y_k)$
- 출력층 가중치: $\partial C / \partial W_2[i,k] = h_i\, \delta_k$
- 은닉층으로 전파: $\partial C / \partial h_i = \sum_k W_2[i,k]\, \delta_k$
- 은닉층 가중치: $\partial C / \partial W_1[i,j] = x_i \cdot \dfrac{\partial C}{\partial h_j}\, h_j (1 - h_j)$

In [ ]:
# ----- 역전파 -----
delta_out = (y - t) * y * (1 - y)          # 출력층 델타
dW2 = np.outer(h, delta_out)               # dC/dW2
db2 = delta_out                            # 편향의 기울기는 델타 그 자체

dh = W2 @ delta_out                        # 오차를 은닉층으로 전달
delta_hid = dh * h * (1 - h)               # 은닉층 델타
dW1 = np.outer(x, delta_hid)               # dC/dW1
db1 = delta_hid

print("delta_out =", delta_out.round(4))
print("dC/dW2 =\n", dW2.round(4))
print("dC/dh  =", dh.round(4))
print("dC/dW1 =\n", dW1.round(4))

같은 계산을 PyTorch의 자동 미분(autograd)에 맡겨 보겠습니다. `requires_grad=True`로 만든 텐서는 계산 과정을 기록하고, `backward()`를 부르면 역전파가 실행됩니다.

In [ ]:
tx, tt = torch.tensor(x), torch.tensor(t)
tW1 = torch.tensor(W1, requires_grad=True)
tW2 = torch.tensor(W2, requires_grad=True)

th = torch.sigmoid(tW1.T @ tx + torch.tensor(b1))
ty = torch.sigmoid(tW2.T @ th + torch.tensor(b2))
tC = 0.5 * ((ty - tt) ** 2).sum()
tC.backward()                      # 역전파: 모든 기울기를 한 번에 계산

print("autograd dC/dW2 =\n", tW2.grad.numpy().round(4))
print("autograd dC/dW1 =\n", tW1.grad.numpy().round(4))
print("손 계산과 일치:", np.allclose(tW1.grad.numpy(), dW1) and np.allclose(tW2.grad.numpy(), dW2))

마지막으로 경사 하강법으로 가중치를 갱신합니다: $w \leftarrow w - \eta\, \partial C/\partial w$

In [ ]:
def forward_cost(W1, b1, W2, b2):
    h = sigmoid(W1.T @ x + b1)
    y = sigmoid(W2.T @ h + b2)
    return 0.5 * np.sum((y - t) ** 2)

eta = 0.5
W1n, b1n = W1 - eta * dW1, b1 - eta * db1
W2n, b2n = W2 - eta * dW2, b2 - eta * db2
print(f"w31: {W2[0,0]:.3f} -> {W2n[0,0]:.3f},  w32: {W2[0,1]:.3f} -> {W2n[0,1]:.3f},  w11: {W1[0,0]:.3f} -> {W1n[0,0]:.3f}")
print(f"cost: {forward_cost(W1, b1, W2, b2):.3f} -> {forward_cost(W1n, b1n, W2n, b2n):.3f}")

# 같은 갱신을 200번 반복
Wa, ba, Wb, bb = W1.copy(), b1.copy(), W2.copy(), b2.copy()
hist = []
for step in range(200):
    h_ = sigmoid(Wa.T @ x + ba); y_ = sigmoid(Wb.T @ h_ + bb)
    hist.append(0.5 * np.sum((y_ - t) ** 2))
    d_out = (y_ - t) * y_ * (1 - y_)
    d_hid = (Wb @ d_out) * h_ * (1 - h_)
    Wb -= eta * np.outer(h_, d_out); bb -= eta * d_out
    Wa -= eta * np.outer(x, d_hid);  ba -= eta * d_hid
plt.figure(figsize=(5, 3)); plt.plot(hist); plt.xlabel("update step"); plt.ylabel("cost C"); plt.show()
print("200번 갱신 후 출력 y =", y_.round(3), "(정답 t = [1, 0])")

**연습 2.** `eta`를 0.05, 5, 50으로 바꿔서 cost 곡선이 어떻게 달라지는지 보세요. 학습률이 너무 작거나 크면 어떻게 됩니까?

---
## Part 3. PyTorch로 와인 분류

강의의 와인 분류 예제를 PyTorch로 다시 만듭니다. 강의 슬라이드는 Keras로 작성되어 있는데, 대응 관계는 아래와 같습니다.

| Keras (슬라이드) | PyTorch (이 노트북) |
|---|---|
| `Dense(20)` | `nn.Linear(13, 20)` |
| `BatchNormalization()` | `nn.BatchNorm1d(20)` |
| `Activation('relu')` | `nn.ReLU()` |
| `Activation('softmax')` + `categorical_crossentropy` | `nn.CrossEntropyLoss()` (softmax 포함) |
| `optimizer='adam'` | `torch.optim.Adam` |
| `model.fit(...)` | 직접 작성한 학습 루프 |

강의와 다른 점이 두 가지 있습니다. (1) 데이터를 train / validation / test 세 덩어리로 나눕니다. (2) 입력을 표준화합니다.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

wine = load_wine()
Xw, yw = wine.data, wine.target
print("X.shape =", Xw.shape, "| y.shape =", yw.shape, "| classes =", np.unique(yw))

# train 60% / validation 15% / test 25%
X_trv, X_te, y_trv, y_te = train_test_split(Xw, yw, test_size=0.25, random_state=SEED, stratify=yw)
X_tr, X_va, y_tr, y_va = train_test_split(X_trv, y_trv, test_size=0.2, random_state=SEED, stratify=y_trv)

# 표준화: 평균과 표준편차는 반드시 train에서만 계산 (test 정보가 새어 들어가지 않도록)
scaler = StandardScaler().fit(X_tr)
to_t = lambda a: torch.tensor(scaler.transform(a), dtype=torch.float32)
Xtr, Xva, Xte = to_t(X_tr), to_t(X_va), to_t(X_te)
ytr, yva, yte = (torch.tensor(a, dtype=torch.long) for a in (y_tr, y_va, y_te))
print("train / val / test =", len(Xtr), len(Xva), len(Xte))

In [ ]:
def make_mlp(n_in=13, n_hidden=20, n_out=3):
    return nn.Sequential(
        nn.Linear(n_in, n_hidden),
        nn.BatchNorm1d(n_hidden),
        nn.ReLU(),
        nn.Linear(n_hidden, n_out),      # 출력은 logit. softmax는 loss 함수 안에서 적용됨
    )

def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        return (model(X).argmax(dim=1) == y).float().mean().item()

def train_classifier(model, optimizer, epochs=30, batch_size=5, seed=SEED):
    # mini-batch 학습 루프: 순전파 -> loss -> 역전파 -> 갱신
    loss_fn = nn.CrossEntropyLoss()
    gen = torch.Generator().manual_seed(seed)
    log = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(Xtr), generator=gen)          # epoch마다 순서를 섞음
        for i in range(0, len(Xtr), batch_size):                # 1 iteration = mini-batch 1개
            idx = perm[i:i + batch_size]
            if len(idx) < 2:                                    # BatchNorm은 샘플 2개 이상 필요
                continue
            optimizer.zero_grad()                               # 이전 기울기 지우기
            loss = loss_fn(model(Xtr[idx]), ytr[idx])           # 순전파 + 비용함수
            loss.backward()                                     # 역전파
            optimizer.step()                                    # 가중치 갱신
        model.eval()
        with torch.no_grad():
            log["train_loss"].append(loss_fn(model(Xtr), ytr).item())
            log["val_loss"].append(loss_fn(model(Xva), yva).item())
        log["train_acc"].append(accuracy(model, Xtr, ytr))
        log["val_acc"].append(accuracy(model, Xva, yva))
    return log

torch.manual_seed(SEED)
model = make_mlp()
print(model)
print("학습 가능한 가중치 수:", sum(p.numel() for p in model.parameters()))
log = train_classifier(model, torch.optim.Adam(model.parameters(), lr=1e-3))

fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].plot(log["train_loss"], label="train"); ax[0].plot(log["val_loss"], label="validation")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("cross-entropy loss"); ax[0].legend()
ax[1].plot(log["train_acc"], label="train"); ax[1].plot(log["val_acc"], label="validation")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("accuracy"); ax[1].legend()
plt.tight_layout(); plt.show()
print(f"test accuracy = {accuracy(model, Xte, yte):.3f}  (test 데이터는 마지막에 한 번만 사용)")

133개가 아니라 106개로 학습했으니 1 epoch의 iteration 수는 `ceil(106 / 5) = 22`입니다.

### 3-1. 학습률의 영향

같은 모델, 같은 초기값에서 학습률만 바꿔 봅니다. optimizer는 기본 SGD입니다.

In [ ]:
plt.figure(figsize=(5.5, 3.4))
for lr in [0.001, 0.05, 3.0]:
    torch.manual_seed(SEED)
    m = make_mlp()
    lg = train_classifier(m, torch.optim.SGD(m.parameters(), lr=lr))
    plt.plot(lg["train_loss"], label=f"lr = {lr}")
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("train loss"); plt.legend(); plt.title("SGD with different learning rates")
plt.show()

### 3-2. Optimizer 비교

학습률을 같게 두고 SGD, Momentum, Adam을 비교합니다.

In [ ]:
optimizers = {
    "SGD":      lambda p: torch.optim.SGD(p, lr=0.005),
    "Momentum": lambda p: torch.optim.SGD(p, lr=0.005, momentum=0.9),
    "Adam":     lambda p: torch.optim.Adam(p, lr=0.005),
}
plt.figure(figsize=(5.5, 3.4))
for name, make_opt in optimizers.items():
    torch.manual_seed(SEED)
    m = make_mlp()
    lg = train_classifier(m, make_opt(m.parameters()))
    plt.plot(lg["train_loss"], label=name)
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("train loss"); plt.legend(); plt.title("Optimizers (lr = 0.005)")
plt.show()

**연습 3.**
1. `batch_size`를 1, 5, 32, 106으로 바꿔 보세요. 가중치 갱신 횟수와 loss 곡선의 모양이 어떻게 달라집니까?
2. `make_mlp`에서 `nn.ReLU()`를 지우면(= 활성화 함수 없이) 정확도가 어떻게 됩니까? 와인 데이터에서는 차이가 작을 수 있습니다. 그 이유를 생각해 보세요.

---
## Part 4. 과적합과 regularization

와인 데이터는 쉬워서 과적합이 잘 보이지 않습니다. 강의 슬라이드처럼 잡음이 섞인 sine 함수 데이터 20개를 큰 신경망으로 학습시켜 보겠습니다.

In [ ]:
rng = np.random.default_rng(3)
f_true = lambda v: np.sin(2 * np.pi * v)
xs_tr = np.sort(rng.uniform(0, 1, 20)); ys_tr = f_true(xs_tr) + rng.normal(0, 0.25, 20)
xs_va = np.sort(rng.uniform(0, 1, 40)); ys_va = f_true(xs_va) + rng.normal(0, 0.25, 40)
T = lambda a: torch.tensor(a, dtype=torch.float32).reshape(-1, 1)
xtr_s, ytr_s, xva_s, yva_s = T(xs_tr), T(ys_tr), T(xs_va), T(ys_va)
xx = T(np.linspace(0, 1, 300))

def big_mlp(dropout=0.0):
    return nn.Sequential(nn.Linear(1, 128), nn.Tanh(), nn.Dropout(dropout),
                         nn.Linear(128, 128), nn.Tanh(), nn.Dropout(dropout),
                         nn.Linear(128, 1))

def train_regressor(model, epochs=4000, lr=0.01, weight_decay=0.0):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    mse = nn.MSELoss(); tr, va = [], []
    best = (np.inf, 0, None)
    for ep in range(epochs):
        model.train(); opt.zero_grad()
        loss = mse(model(xtr_s), ytr_s); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            tr.append(mse(model(xtr_s), ytr_s).item()); v = mse(model(xva_s), yva_s).item(); va.append(v)
            if v < best[0]:
                best = (v, ep, model(xx).numpy().ravel())      # validation이 가장 좋았던 시점의 예측 저장
    with torch.no_grad():
        final = model(xx).numpy().ravel()
    return tr, va, best, final

torch.manual_seed(SEED)
tr, va, best, final = train_regressor(big_mlp())

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(tr, label="train"); ax[0].plot(va, label="validation")
ax[0].axvline(best[1], color="gray", ls="--"); ax[0].text(best[1] * 1.3, max(va) * 0.5, "early stopping", color="gray")
ax[0].set_xscale("log"); ax[0].set_yscale("log"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("MSE"); ax[0].legend()
ax[1].plot(xx, f_true(xx.numpy()), "k--", lw=1, label="true function")
ax[1].plot(xx, final, label=f"last epoch (val MSE {va[-1]:.3f})")
ax[1].plot(xx, best[2], label=f"early stopping, epoch {best[1]} (val MSE {best[0]:.3f})")
ax[1].scatter(xs_tr, ys_tr, c="k", s=20, zorder=3, label="train data")
ax[1].set_ylim(-2, 2); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

train loss는 계속 줄지만 validation loss는 어느 시점부터 다시 올라갑니다. 마지막 epoch의 모델은 학습 데이터의 잡음까지 따라갑니다.

이번에는 weight decay와 dropout을 넣어 봅니다.

In [ ]:
settings = {
    "no regularization":      dict(dropout=0.0, weight_decay=0.0),
    "weight decay = 1e-3":    dict(dropout=0.0, weight_decay=1e-3),
    "dropout = 0.2":          dict(dropout=0.2, weight_decay=0.0),
}
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for name, s in settings.items():
    torch.manual_seed(SEED)
    tr, va, best, final = train_regressor(big_mlp(s["dropout"]), weight_decay=s["weight_decay"])
    ax[0].plot(va, label=name)
    ax[1].plot(xx, final, label=f"{name} (val MSE {va[-1]:.3f})")
ax[0].set_xscale("log"); ax[0].set_yscale("log"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("validation MSE"); ax[0].legend(fontsize=8)
ax[1].plot(xx, f_true(xx.numpy()), "k--", lw=1); ax[1].scatter(xs_tr, ys_tr, c="k", s=20, zorder=3)
ax[1].set_ylim(-2, 2); ax[1].legend(fontsize=8); ax[1].set_title("prediction at last epoch")
plt.tight_layout(); plt.show()

**연습 4.** 학습 데이터 수를 20개에서 200개로 늘리면(`rng.uniform(0, 1, 200)`) 과적합이 어떻게 달라집니까?

---
## Part 5. 결정 그래프 신경망(CGCNN) 직접 만들기

이제 결정 구조를 그래프로 바꾸고, CGCNN을 PyTorch로 직접 구현해 학습시킵니다.

### 사용하는 데이터에 대한 주의

실제 연구에서는 Materials Project 같은 DFT 데이터베이스를 씁니다. 여기서는 인터넷 연결이나 API key 없이 바로 실행할 수 있도록 **가상의 데이터**를 만들어 씁니다.

- 구조: rocksalt, CsCl, zincblende형 AB 화합물과 입방 perovskite ABX₃. 격자 상수를 조금씩 바꾼 구조를 포함합니다.
- 타깃: 원소의 전기음성도와 반지름으로 정한 **간단한 pair potential(Morse 형태)로 계산한 원자당 에너지**입니다. DFT 값이 아니고 실제 물질의 에너지와 무관합니다.

목적은 "구조 → 그래프 → message passing → 물성 예측"의 전 과정을 직접 따라가 보는 것입니다. 실제 데이터로 바꾸려면 구조와 타깃을 만드는 부분만 교체하면 됩니다.

In [ ]:
# 원소 정보: 원자번호, 족, 주기, Pauling 전기음성도, 공유 반지름(Å). 실습용 근사값
ELEMENTS = {
    "Li": (3, 1, 2, 0.98, 1.28),  "Na": (11, 1, 3, 0.93, 1.66), "K":  (19, 1, 4, 0.82, 2.03),
    "Rb": (37, 1, 5, 0.82, 2.20), "Cs": (55, 1, 6, 0.79, 2.44),
    "Mg": (12, 2, 3, 1.31, 1.41), "Ca": (20, 2, 4, 1.00, 1.76), "Sr": (38, 2, 5, 0.95, 1.95), "Ba": (56, 2, 6, 0.89, 2.15),
    "Sn": (50, 14, 5, 1.96, 1.39), "Pb": (82, 14, 6, 2.33, 1.46),
    "O":  (8, 16, 2, 3.44, 0.66),  "S":  (16, 16, 3, 2.58, 1.05),
    "F":  (9, 17, 2, 3.98, 0.57),  "Cl": (17, 17, 3, 3.16, 1.02), "Br": (35, 17, 4, 2.96, 1.20), "I": (53, 17, 5, 2.66, 1.39),
}
CHI = {k: v[3] for k, v in ELEMENTS.items()}
RAD = {k: v[4] for k, v in ELEMENTS.items()}

def make_structure(prototype, species, scale=1.0):
    # (격자 벡터 3x3 [Å], 분수 좌표 Nx3, 원소 리스트)를 돌려준다
    if prototype == "perovskite":                       # ABX3, 단순 입방, 원자 5개
        A, B, X = species
        a = 2 * (RAD[B] + RAD[X]) * scale
        lattice = a * np.eye(3)
        frac = [[0, 0, 0], [.5, .5, .5], [.5, .5, 0], [.5, 0, .5], [0, .5, .5]]
        return lattice, np.array(frac, float), [A, B, X, X, X]
    A, X = species
    d = RAD[A] + RAD[X]                                 # 가장 가까운 A-X 거리의 기준값
    fcc = np.array([[0, .5, .5], [.5, 0, .5], [.5, .5, 0]])
    if prototype == "rocksalt":
        a = 2 * d * scale;              lattice, fx = a * fcc, [.5, .5, .5]
    elif prototype == "zincblende":
        a = 4 * d / np.sqrt(3) * scale; lattice, fx = a * fcc, [.25, .25, .25]
    elif prototype == "CsCl":
        a = 2 * d / np.sqrt(3) * scale; lattice, fx = a * np.eye(3), [.5, .5, .5]
    return lattice, np.array([[0, 0, 0], fx], float), [A, X]

lattice, frac, species = make_structure("rocksalt", ("Na", "Cl"))
print("NaCl (rocksalt) primitive cell\nlattice =\n", lattice.round(3), "\nfrac =\n", frac, "\nspecies =", species)

### 5-1. 결정을 그래프로: 주기 경계 조건에서 이웃 찾기

결정은 단위 셀이 무한히 반복됩니다. 그래서 이웃을 찾을 때 옆 셀에 있는 원자(주기 이미지)까지 포함해야 합니다. 같은 원자 쌍 사이에 edge가 여러 개 생길 수 있고, 자기 자신의 이미지와도 연결될 수 있습니다.

CGCNN 공식 코드의 기본 설정과 같이 **반경 8 Å 이내, 가까운 순서로 최대 12개**를 이웃으로 잡습니다.

In [ ]:
RADIUS, MAX_NBR = 8.0, 12

def all_neighbors(lattice, frac, radius=RADIUS):
    # 각 원자 i에 대해 반경 안의 모든 (j, 거리) 목록. 주기 이미지 포함
    cart = frac @ lattice
    vol = abs(np.linalg.det(lattice))
    heights = vol / np.linalg.norm(np.cross(lattice[[1, 2, 0]], lattice[[2, 0, 1]]), axis=1)   # 마주 보는 면 사이 거리
    n = np.ceil(radius / heights).astype(int)                                                  # 방향별 필요한 이미지 수
    shifts = np.array([[i, j, k] for i in range(-n[0], n[0] + 1)
                                 for j in range(-n[1], n[1] + 1)
                                 for k in range(-n[2], n[2] + 1)]) @ lattice
    out = []
    for i in range(len(cart)):
        diff = cart[None, :, :] + shifts[:, None, :] - cart[i]       # [이미지, j, 3]
        dist = np.linalg.norm(diff, axis=2)
        mask = (dist < radius) & (dist > 1e-8)                       # 자기 자신(거리 0)은 제외
        img, j = np.nonzero(mask)
        out.append((j, dist[img, j]))
    return out

def build_graph(lattice, frac, elems, max_nbr=MAX_NBR):
    # 이웃 인덱스 [N, M]와 거리 [N, M]. 가까운 순서로 M개
    # 거리가 같은 이웃이 여러 개면 원자번호 순으로 고른다 (회전하거나 원자 순서를 바꿔도 같은 선택이 되도록)
    Z = np.array([ELEMENTS[s][0] for s in elems])
    idx, dst = [], []
    for j, d in all_neighbors(lattice, frac):
        order = np.lexsort((Z[j], np.round(d, 4)))[:max_nbr]
        assert len(order) == max_nbr, "이웃 수가 부족합니다. RADIUS를 키우세요."
        idx.append(j[order]); dst.append(d[order])
    return np.array(idx), np.array(dst)

nbr_idx, nbr_dist = build_graph(lattice, frac, species)
print("Na(0번 원자)의 이웃 원자 번호:", nbr_idx[0])
print("Na(0번 원자)의 이웃까지 거리 :", nbr_dist[0].round(3))

NaCl에서 Na의 가장 가까운 이웃은 Cl(1번 원자) 6개이고, 모두 같은 거리입니다. 그다음 이웃은 Na(0번, 자기 자신의 주기 이미지)입니다. primitive cell에 원자가 2개뿐이어도 그래프에는 이렇게 많은 edge가 생깁니다.

### 5-2. 타깃 만들기 (가상의 에너지)

원자 쌍마다 Morse 형태의 pair potential을 더해서 원자당 에너지를 계산합니다. 전기음성도 차이가 클수록 결합이 강하고, 평형 거리는 두 반지름의 합입니다.

In [ ]:
def toy_energy(lattice, frac, species):
    # 가상의 원자당 에너지 (eV/atom 단위로 가정). DFT 값이 아님
    total = 0.0
    for i, (j, d) in enumerate(all_neighbors(lattice, frac)):
        si = species[i]
        chi_j = np.array([CHI[species[k]] for k in j]); rad_j = np.array([RAD[species[k]] for k in j])
        depth = 0.03 + 0.12 * np.abs(CHI[si] - chi_j)                 # 결합 세기
        r0 = RAD[si] + rad_j                                          # 평형 거리
        total += 0.5 * np.sum(depth * ((1 - np.exp(-1.5 * (d - r0))) ** 2 - 1))
    return total / len(species)

scales = np.linspace(0.85, 1.25, 30)
plt.figure(figsize=(5, 3.2))
for proto in ["rocksalt", "CsCl", "zincblende"]:
    plt.plot(scales, [toy_energy(*make_structure(proto, ("Na", "Cl"), s)) for s in scales], label=proto)
plt.xlabel("lattice scale"); plt.ylabel("toy energy (eV/atom)"); plt.title("NaCl"); plt.legend(); plt.show()

In [ ]:
# 데이터셋 생성
CATIONS = ["Li", "Na", "K", "Rb", "Cs", "Mg", "Ca", "Sr", "Ba"]
ANIONS  = ["F", "Cl", "Br", "I", "O", "S"]
SCALES  = np.linspace(0.92, 1.14, 8)

records = []
t0 = time.time()
for A in CATIONS:
    for X in ANIONS:
        for proto in ["rocksalt", "CsCl", "zincblende"]:
            for s in SCALES:
                records.append(dict(prototype=proto, species=(A, X), scale=s))
for A in ["K", "Rb", "Cs"]:
    for B in ["Ca", "Sr", "Ba", "Sn", "Pb"]:
        for X in ["F", "Cl", "Br", "I"]:
            for s in SCALES:
                records.append(dict(prototype="perovskite", species=(A, B, X), scale=s))

for r in records:
    lat, fr, sp = make_structure(r["prototype"], r["species"], r["scale"])
    r.update(lattice=lat, frac=fr, elems=sp, energy=toy_energy(lat, fr, sp))
    r["nbr_idx"], r["nbr_dist"] = build_graph(lat, fr, sp)
    r["formula"] = "".join(r["species"]) + ("3" if r["prototype"] == "perovskite" else "")

energies = np.array([r["energy"] for r in records])
print(f"구조 {len(records)}개 생성 ({time.time() - t0:.1f}초)")
print(f"에너지 범위: {energies.min():.2f} ~ {energies.max():.2f} eV/atom, 평균 {energies.mean():.2f}")
plt.figure(figsize=(5, 2.8)); plt.hist(energies, bins=40); plt.xlabel("toy energy (eV/atom)"); plt.ylabel("count"); plt.show()

### 5-3. Node와 edge의 특성 벡터

**Node(원자) 특성**: 강의의 `atom_init.json` 슬라이드와 같은 방식입니다. 족과 주기는 one-hot으로, 전기음성도와 반지름은 10개 구간으로 나눠 one-hot으로 만들어 이어 붙입니다.

**Edge(결합) 특성**: 원자 사이 거리를 Gaussian 기저로 전개합니다. 0–8 Å 구간에 0.2 Å 간격으로 Gaussian 41개를 놓습니다.

In [ ]:
def onehot(index, size):
    v = np.zeros(size); v[index] = 1.0; return v

def bin_index(value, lo, hi, n=10):
    return int(np.clip((value - lo) / (hi - lo) * n, 0, n - 1))

def atom_feature(symbol):
    Z, group, period, chi, rad = ELEMENTS[symbol]
    return np.concatenate([onehot(group - 1, 18), onehot(period - 1, 6),
                           onehot(bin_index(chi, 0.5, 4.0), 10), onehot(bin_index(rad, 0.5, 2.5), 10)])

GAUSS_CENTERS = np.arange(0, RADIUS + 1e-9, 0.2)
def gaussian_expand(dist, width=0.2):
    return np.exp(-((dist[..., None] - GAUSS_CENTERS) ** 2) / width ** 2)

print("Na 특성 벡터 (44차원):", atom_feature("Na").astype(int))
print("Cl 특성 벡터 (44차원):", atom_feature("Cl").astype(int))

d_NaCl = nbr_dist[0, 0]
plt.figure(figsize=(6, 2.6))
plt.stem(GAUSS_CENTERS, gaussian_expand(np.array(d_NaCl)), basefmt=" ")
plt.axvline(d_NaCl, color="r"); plt.xlabel("Gaussian center (Å)"); plt.ylabel("edge feature value")
plt.title(f"Na-Cl distance {d_NaCl:.2f} Å -> {len(GAUSS_CENTERS)}-dim vector"); plt.show()

for r in records:
    r["atom_fea"] = np.array([atom_feature(s) for s in r["elems"]])
    r["nbr_fea"] = gaussian_expand(r["nbr_dist"])
print("atom_fea:", records[0]["atom_fea"].shape, "| nbr_fea:", records[0]["nbr_fea"].shape, "| nbr_idx:", records[0]["nbr_idx"].shape)

### 5-4. CGCNN 모델

CGCNN의 convolution 식입니다(강의 「CGCNN convolution」 슬라이드).

$$z_{(i,j)_k} = v_i \oplus v_j \oplus u_{(i,j)_k}$$

$$v_i \leftarrow v_i + \sum_{j,k} \underbrace{\sigma\!\left(z W_f + b_f\right)}_{\text{gate}} \odot \underbrace{g\!\left(z W_s + b_s\right)}_{\text{message}}$$

아래 `ConvLayer`의 각 줄이 이 식의 어느 부분인지 주석으로 표시했습니다. 구조는 CGCNN 공식 코드를 따랐습니다.

In [ ]:
class ConvLayer(nn.Module):
    def __init__(self, atom_len, nbr_len):
        super().__init__()
        self.atom_len = atom_len
        self.fc = nn.Linear(2 * atom_len + nbr_len, 2 * atom_len)    # W_f와 W_s를 한 번에 계산
        self.bn1 = nn.BatchNorm1d(2 * atom_len)
        self.bn2 = nn.BatchNorm1d(atom_len)
        self.softplus = nn.Softplus()

    def forward(self, v, u, nbr_idx):
        # v: [N, F] 원자 특성,  u: [N, M, G] 결합 특성,  nbr_idx: [N, M] 이웃 원자 번호
        N, M = nbr_idx.shape
        v_j = v[nbr_idx]                                             # [N, M, F] 이웃 원자의 특성
        v_i = v.unsqueeze(1).expand(N, M, self.atom_len)             # [N, M, F] 중심 원자의 특성
        z = torch.cat([v_i, v_j, u], dim=2)                          # z = v_i ⊕ v_j ⊕ u
        z = self.bn1(self.fc(z).view(N * M, -1)).view(N, M, -1)
        gate, message = z.chunk(2, dim=2)
        gate, message = torch.sigmoid(gate), self.softplus(message)  # σ(zW_f + b_f),  g(zW_s + b_s)
        summed = self.bn2((gate * message).sum(dim=1))               # 이웃에 대한 합
        return self.softplus(v + summed)                             # v_i + ... (residual)


class CGCNN(nn.Module):
    def __init__(self, orig_atom_len, nbr_len, atom_len=64, n_conv=3, h_len=64):
        super().__init__()
        self.embedding = nn.Linear(orig_atom_len, atom_len)
        self.convs = nn.ModuleList([ConvLayer(atom_len, nbr_len) for _ in range(n_conv)])
        self.fc = nn.Sequential(nn.Softplus(), nn.Linear(atom_len, h_len), nn.Softplus(), nn.Linear(h_len, 1))

    def atom_vectors(self, atom_fea, nbr_fea, nbr_idx):
        v = self.embedding(atom_fea)
        for conv in self.convs:                                      # message passing을 n_conv번 반복
            v = conv(v, nbr_fea, nbr_idx)
        return v

    def forward(self, atom_fea, nbr_fea, nbr_idx, batch):
        v = self.atom_vectors(atom_fea, nbr_fea, nbr_idx)
        n_crystal = int(batch.max()) + 1                             # pooling: 결정별로 원자 특성의 평균
        pooled = torch.zeros(n_crystal, v.shape[1]).index_add_(0, batch, v)
        pooled = pooled / torch.bincount(batch, minlength=n_crystal).unsqueeze(1)
        return self.fc(pooled).squeeze(1)

결정마다 원자 수가 다르므로, mini-batch를 만들 때 여러 결정의 원자를 한 줄로 이어 붙이고 "각 원자가 몇 번째 결정에 속하는지"를 `batch` 벡터에 기록합니다. 이웃 번호는 이어 붙인 뒤의 번호로 바꿔 줍니다.

In [ ]:
def collate(recs):
    atom_fea, nbr_fea, nbr_idx, batch, target = [], [], [], [], []
    offset = 0
    for b, r in enumerate(recs):
        n = len(r["atom_fea"])
        atom_fea.append(r["atom_fea"]); nbr_fea.append(r["nbr_fea"])
        nbr_idx.append(r["nbr_idx"] + offset)              # 원자 번호를 batch 전체 기준으로 이동
        batch += [b] * n; target.append(r["energy"]); offset += n
    f32 = lambda a: torch.tensor(np.concatenate(a), dtype=torch.float32)
    return (f32(atom_fea), f32(nbr_fea), torch.tensor(np.concatenate(nbr_idx), dtype=torch.long),
            torch.tensor(batch, dtype=torch.long)), torch.tensor(target, dtype=torch.float32)

def predict(model, recs, mean, std):
    model.eval()
    with torch.no_grad():
        inputs, _ = collate(recs)
        return (model(*inputs) * std + mean).numpy()

def train_gnn(train, val, epochs=40, batch_size=64, lr=5e-3, n_conv=3, seed=SEED, verbose=True):
    torch.manual_seed(seed)
    model = CGCNN(orig_atom_len=train[0]["atom_fea"].shape[1], nbr_len=train[0]["nbr_fea"].shape[-1], n_conv=n_conv)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)      # 학습률을 점점 줄임
    y_train = np.array([r["energy"] for r in train])
    mean, std = float(y_train.mean()), float(y_train.std())                    # 타깃 표준화 (train 기준)
    mae = lambda recs: float(np.mean(np.abs(predict(model, recs, mean, std) - np.array([r["energy"] for r in recs]))))
    rng = np.random.default_rng(seed); log = {"train": [], "val": []}
    best_val, best_state = np.inf, None
    for ep in range(epochs):
        model.train()
        order = rng.permutation(len(train))
        for i in range(0, len(train), batch_size):
            inputs, target = collate([train[k] for k in order[i:i + batch_size]])
            opt.zero_grad()
            loss = nn.functional.mse_loss(model(*inputs), (target - mean) / std)
            loss.backward(); opt.step()
        sched.step()
        log["train"].append(mae(train)); log["val"].append(mae(val))
        if log["val"][-1] < best_val:                                          # validation이 가장 좋은 모델을 보관
            best_val = log["val"][-1]; best_state = {k: v.clone() for k, v in model.state_dict().items()}
        if verbose and (ep + 1) % 10 == 0:
            print(f"epoch {ep + 1:3d} | train MAE {log['train'][-1]:.4f} | val MAE {log['val'][-1]:.4f}")
    model.load_state_dict(best_state)
    return model, (mean, std), log

### 5-5. 학습과 평가

먼저 무작위로 train 70% / validation 15% / test 15%로 나눠 학습합니다.

In [ ]:
perm = np.random.default_rng(SEED).permutation(len(records))
n_tr, n_va = int(0.7 * len(records)), int(0.15 * len(records))
train_set = [records[i] for i in perm[:n_tr]]
val_set   = [records[i] for i in perm[n_tr:n_tr + n_va]]
test_set  = [records[i] for i in perm[n_tr + n_va:]]
print("train / val / test =", len(train_set), len(val_set), len(test_set))

t0 = time.time()
gnn, norm, log = train_gnn(train_set, val_set)
print(f"학습 시간 {time.time() - t0:.0f}초 | 가중치 수 {sum(p.numel() for p in gnn.parameters()):,}")

y_test = np.array([r["energy"] for r in test_set])
p_test = predict(gnn, test_set, *norm)
mae_gnn = np.mean(np.abs(p_test - y_test))

fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.6))
ax[0].plot(log["train"], label="train"); ax[0].plot(log["val"], label="validation")
ax[0].set_yscale("log"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("MAE (eV/atom)"); ax[0].legend()
lim = [y_test.min() - 0.1, y_test.max() + 0.1]
ax[1].scatter(y_test, p_test, s=10, alpha=0.6); ax[1].plot(lim, lim, "k--", lw=1)
ax[1].set_xlabel("target (toy energy)"); ax[1].set_ylabel("CGCNN prediction"); ax[1].set_title(f"test MAE = {mae_gnn:.4f} eV/atom")
plt.tight_layout(); plt.show()

### 5-6. 구조 정보가 왜 필요한가: 조성만 쓰는 모델과 비교

같은 데이터를 **조성 정보만** 쓰는 MLP로 학습시켜 봅니다. 입력은 원자 특성 벡터의 평균입니다. 격자 상수나 구조 prototype에 대한 정보는 들어가지 않습니다.

In [ ]:
comp = lambda recs: torch.tensor(np.array([r["atom_fea"].mean(axis=0) for r in recs]), dtype=torch.float32)
tgt  = lambda recs: torch.tensor([r["energy"] for r in recs], dtype=torch.float32)
Xc_tr, Xc_va, Xc_te = comp(train_set), comp(val_set), comp(test_set)
yc_tr, yc_va = tgt(train_set), tgt(val_set)
c_mean, c_std = yc_tr.mean(), yc_tr.std()

torch.manual_seed(SEED)
mlp = nn.Sequential(nn.Linear(Xc_tr.shape[1], 64), nn.Softplus(), nn.Linear(64, 64), nn.Softplus(), nn.Linear(64, 1))
opt = torch.optim.Adam(mlp.parameters(), lr=3e-3)
best_val, best_pred = np.inf, None
for ep in range(1500):
    opt.zero_grad()
    nn.functional.mse_loss(mlp(Xc_tr).squeeze(1), (yc_tr - c_mean) / c_std).backward(); opt.step()
    with torch.no_grad():
        v = (mlp(Xc_va).squeeze(1) * c_std + c_mean - yc_va).abs().mean().item()
        if v < best_val:
            best_val, best_pred = v, (mlp(Xc_te).squeeze(1) * c_std + c_mean).numpy()
mae_comp = np.mean(np.abs(best_pred - y_test))

fig, ax = plt.subplots(1, 2, figsize=(9, 3.6), sharex=True, sharey=True)
for a, (p, name, m) in zip(ax, [(best_pred, "composition-only MLP", mae_comp), (p_test, "CGCNN (structure)", mae_gnn)]):
    a.scatter(y_test, p, s=10, alpha=0.6); a.plot(lim, lim, "k--", lw=1)
    a.set_title(f"{name}\ntest MAE = {m:.4f} eV/atom"); a.set_xlabel("target")
ax[0].set_ylabel("prediction"); plt.tight_layout(); plt.show()

조성만 쓰는 모델에서는 점들이 세로로 늘어섭니다. 조성이 같으면 구조가 달라도 같은 값을 예측할 수밖에 없기 때문입니다.

### 5-7. 대칭성 확인

그래프 표현을 쓰면 다음 변환에 대해 예측이 변하지 않아야 합니다.

1. 원자 번호 순서를 바꾼다 (순열 불변성)
2. 결정을 통째로 회전시킨다 (회전 불변성)
3. 같은 결정을 더 큰 셀(supercell)로 표현한다

In [ ]:
def predict_structure(model, norm, lattice, frac, elems):
    idx, dst = build_graph(lattice, frac, elems)
    rec = dict(atom_fea=np.array([atom_feature(s) for s in elems]), nbr_fea=gaussian_expand(dst), nbr_idx=idx, energy=0.0)
    return float(predict(model, [rec], *norm)[0])

lat, fr, el = make_structure("perovskite", ("Cs", "Pb", "I"), 1.0)
e_ref = predict_structure(gnn, norm, lat, fr, el)

# 1) 원자 순서 바꾸기
order = [4, 2, 0, 3, 1]
e_perm = predict_structure(gnn, norm, lat, fr[order], [el[i] for i in order])

# 2) 임의의 회전 행렬 적용 (QR 분해로 직교 행렬 생성)
Q, _ = np.linalg.qr(np.random.default_rng(1).normal(size=(3, 3)))
e_rot = predict_structure(gnn, norm, lat @ Q, fr, el)

# 3) 2x1x1 supercell
lat2 = lat * np.array([[2], [1], [1]])
fr2 = np.concatenate([fr * [0.5, 1, 1], fr * [0.5, 1, 1] + [0.5, 0, 0]])
e_super = predict_structure(gnn, norm, lat2, fr2, el + el)

print(f"CsPbI3 예측값 (원래 구조)   : {e_ref:.5f}")
print(f"원자 순서 변경             : {e_perm:.5f}")
print(f"회전                      : {e_rot:.5f}")
print(f"2x1x1 supercell (원자 10개): {e_super:.5f}")
print(f"참고: 타깃 값              : {toy_energy(lat, fr, el):.5f}")

세 경우 모두 예측값이 (수치 오차 범위에서) 같습니다. 모델이 이것을 데이터에서 배운 것이 아니라, 거리만 입력으로 쓰고 합과 평균으로 정보를 모으는 **구조 자체가 보장**하는 성질입니다.

주의할 점이 하나 있습니다. "가까운 순서로 12개"처럼 이웃 수를 제한하면, 같은 거리에 있는 이웃 중 일부만 뽑히는 경우가 생깁니다. 이때 무엇을 뽑을지가 원자 순서나 좌표의 미세한 오차에 따라 달라지면 불변성이 깨집니다. `build_graph`에서 거리가 같으면 원자번호 순으로 고르도록 한 것이 이 때문입니다. 최근 모델들은 이웃 수를 제한하지 않고 cutoff 반경 안의 이웃을 모두 쓰는 방식이 일반적입니다.

### 5-8. 모델이 배운 원자 표현 들여다보기

convolution을 거친 원자 특성 벡터는 원소의 종류뿐 아니라 주변 환경을 반영합니다. 입력 단계에서는 모든 Cl 원자가 똑같은 44차원 벡터였습니다. 3번의 convolution 뒤에 Cl 원자의 벡터가 구조 유형, 상대 원소, 격자 상수에 따라 얼마나 달라졌는지 PCA로 2차원에 그려 봅니다.

In [ ]:
from sklearn.decomposition import PCA

vecs, labels = [], []
gnn.eval()
with torch.no_grad():
    for r in records:
        if "Cl" in r["elems"]:
            (atom_fea, nbr_fea, nbr_idx, batch), _ = collate([r])
            v = gnn.atom_vectors(atom_fea, nbr_fea, nbr_idx).numpy()
            for k, s in enumerate(r["elems"]):
                if s == "Cl":
                    vecs.append(v[k]); labels.append(r["prototype"]); break
xy = PCA(n_components=2).fit_transform(np.array(vecs)); labels = np.array(labels)
plt.figure(figsize=(5.5, 4))
for proto in ["rocksalt", "CsCl", "zincblende", "perovskite"]:
    m = labels == proto
    plt.scatter(xy[m, 0], xy[m, 1], s=14, alpha=0.7, label=proto)
plt.xlabel("PC 1"); plt.ylabel("PC 2"); plt.title("Cl atom vectors after 3 conv layers"); plt.legend(); plt.show()

**연습 5.**
1. `train_gnn(train_set, val_set, n_conv=1)`처럼 convolution 층 수를 1, 2, 3, 5로 바꿔 test MAE를 비교해 보세요.
2. `MAX_NBR`을 12에서 6으로 줄이면 어떻게 됩니까? (바꾼 뒤 5-1부터 다시 실행) rocksalt에서 가장 가까운 이웃 6개만 보면 어떤 정보를 놓칩니까?
3. `gaussian_expand`를 쓰지 않고 거리 값 하나만 edge 특성으로 넣으면(`r["nbr_fea"] = r["nbr_dist"][..., None]`) 성능이 어떻게 달라집니까?

---
## Part 6. 데이터 분할과 누출

Part 5의 test MAE는 무작위 분할로 얻은 값입니다. test에 있는 구조와 **조성도 prototype도 같고 격자 상수만 조금 다른 구조**가 train에 들어 있습니다.

실제로 알고 싶은 것은 "학습 때 한 번도 보지 못한 화학계에서 얼마나 맞는가"입니다. 요오드(I)를 포함한 구조를 전부 test로 빼고 다시 학습해 봅니다. 모델은 I가 들어간 구조를 하나도 보지 못한 채 요오드화물의 에너지를 예측해야 합니다.

In [ ]:
has_I = np.array(["I" in r["elems"] for r in records])
group_test = [r for r, m in zip(records, has_I) if m]
rest = [r for r, m in zip(records, has_I) if not m]
perm_g = np.random.default_rng(SEED).permutation(len(rest))
n_val_g = int(0.15 * len(rest))
group_val = [rest[i] for i in perm_g[:n_val_g]]
group_train = [rest[i] for i in perm_g[n_val_g:]]
print("group split: train / val / test(iodides) =", len(group_train), len(group_val), len(group_test))

gnn_g, norm_g, log_g = train_gnn(group_train, group_val, verbose=False)
y_g = np.array([r["energy"] for r in group_test]); p_g = predict(gnn_g, group_test, *norm_g)
mae_group = np.mean(np.abs(p_g - y_g))

# 비교: 무작위 분할 모델(Part 5)이 test 안의 요오드화물을 맞히는 정도
rand_I = [r for r in test_set if "I" in r["elems"]]
mae_rand_I = np.mean(np.abs(predict(gnn, rand_I, *norm) - np.array([r["energy"] for r in rand_I])))

fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.6))
ax[0].bar(["random split\n(iodides in test)", "group split\n(all iodides held out)"], [mae_rand_I, mae_group], color=["#4472c4", "#c0504d"])
ax[0].set_ylabel("MAE on iodides (eV/atom)")
for i, v in enumerate([mae_rand_I, mae_group]):
    ax[0].text(i, v, f"{v:.4f}", ha="center", va="bottom")
lim_g = [min(y_g.min(), p_g.min()) - 0.1, max(y_g.max(), p_g.max()) + 0.1]
ax[1].scatter(y_g, p_g, s=10, alpha=0.6, color="#c0504d"); ax[1].plot(lim_g, lim_g, "k--", lw=1)
ax[1].set_xlabel("target"); ax[1].set_ylabel("prediction"); ax[1].set_title("group split: unseen iodides")
plt.tight_layout(); plt.show()
print(f"무작위 분할에서의 요오드화물 MAE: {mae_rand_I:.4f}")
print(f"그룹 분할에서의 요오드화물 MAE  : {mae_group:.4f}  ({mae_group / mae_rand_I:.0f}배)")

같은 모델 구조, 같은 종류의 데이터인데도 분할 방법에 따라 오차가 크게 달라집니다. 무작위 분할의 MAE는 "비슷한 것을 이미 본 상태"에서의 성능이고, 새로운 화학계를 탐색할 때의 성능이 아닙니다.

여기서는 원자 특성을 족, 주기, 전기음성도, 반지름으로 만들었기 때문에, 모델이 I를 한 번도 보지 못했어도 "17족, 5주기 원소"라는 정보로 어느 정도 외삽할 수 있습니다. 원소마다 독립적인 벡터를 학습시키는 방식(embedding)이었다면 학습 때 없던 원소에 대해서는 아무 정보도 없었을 것입니다.

**연습 6.**
1. I 대신 perovskite 구조 전체를 test로 빼 보세요(`r["prototype"] == "perovskite"`). 처음 보는 구조 유형에 대한 외삽은 얼마나 됩니까?
2. 논문에서 "test MAE"를 볼 때 어떤 질문을 해야 할지 정리해 보세요.

---
## Part 7. (선택) 사전학습된 범용 퍼텐셜 CHGNet 사용

지금까지는 가상의 에너지를 맞히는 작은 모델을 직접 학습시켰습니다. 실제 연구에서는 Materials Project의 DFT 계산 약 158만 구조로 미리 학습된 범용 원자간 퍼텐셜을 내려받아 바로 쓸 수 있습니다.

여기서는 CHGNet으로 (1) 에너지-부피 곡선을 그리고 (2) 구조를 이완시켜 봅니다. DFT 계산 없이 몇 초면 끝납니다.

`pip install chgnet ase`가 필요합니다. 설치하지 않았다면 이 부분은 건너뛰어도 됩니다. 학습된 가중치는 `chgnet` 패키지에 포함되어 있습니다.

In [ ]:
try:
    import warnings; warnings.filterwarnings("ignore")
    from chgnet.model import CHGNet, StructOptimizer
    from pymatgen.core import Structure, Lattice
    chgnet = CHGNet.load()
    HAS_CHGNET = True
    print("CHGNet 가중치 수:", f"{sum(p.numel() for p in chgnet.parameters()):,}")
except ImportError as e:
    HAS_CHGNET = False
    print("chgnet이 설치되어 있지 않아 Part 7을 건너뜁니다:", e)

In [ ]:
if HAS_CHGNET:
    # NaCl (rocksalt) conventional cell, 격자 상수를 바꿔 가며 에너지 예측
    a_list = np.linspace(5.2, 6.2, 11)
    e_list = []
    for a in a_list:
        s = Structure.from_spacegroup("Fm-3m", Lattice.cubic(a), ["Na", "Cl"], [[0, 0, 0], [0.5, 0.5, 0.5]])
        e_list.append(float(chgnet.predict_structure(s)["e"]))        # eV/atom
    coef = np.polyfit(a_list, e_list, 3)
    a_fine = np.linspace(a_list[0], a_list[-1], 400)
    a_min = a_fine[np.argmin(np.polyval(coef, a_fine))]
    plt.figure(figsize=(5, 3.2))
    plt.plot(a_list, e_list, "o"); plt.plot(a_fine, np.polyval(coef, a_fine))
    plt.axvline(a_min, color="gray", ls="--")
    plt.xlabel("lattice constant a (Å)"); plt.ylabel("CHGNet energy (eV/atom)"); plt.title(f"NaCl: minimum at a = {a_min:.2f} Å")
    plt.show()
    print("NaCl 격자 상수의 실험값은 약 5.64 Å입니다. PBE로 학습한 모델이므로 실험값보다 약간 크게 나오는 것이 정상입니다.")

In [ ]:
if HAS_CHGNET:
    # 일부러 찌그러뜨린 구조에서 출발해 이완
    s0 = Structure.from_spacegroup("Fm-3m", Lattice.cubic(6.0), ["Na", "Cl"], [[0, 0, 0], [0.5, 0.5, 0.5]])
    s0.perturb(0.15)                                   # 원자를 무작위로 0.15 Å씩 이동
    pred0 = chgnet.predict_structure(s0)
    print(f"이완 전: a = {s0.lattice.a:.3f} Å, E = {float(pred0['e']):.4f} eV/atom, 최대 힘 = {np.abs(pred0['f']).max():.3f} eV/Å")

    t0 = time.time()
    result = StructOptimizer(model=chgnet).relax(s0, fmax=0.05, verbose=False)
    s1 = result["final_structure"]; pred1 = chgnet.predict_structure(s1)
    print(f"이완 후: a = {s1.lattice.a:.3f} Å, E = {float(pred1['e']):.4f} eV/atom, 최대 힘 = {np.abs(pred1['f']).max():.3f} eV/Å")
    print(f"이완 단계 수 {len(result['trajectory'].energies)}, 소요 시간 {time.time() - t0:.1f}초")

    plt.figure(figsize=(5, 3))
    plt.plot(np.array(result["trajectory"].energies) / len(s0), "o-")
    plt.xlabel("relaxation step"); plt.ylabel("energy (eV/atom)"); plt.show()

**연습 7.**
1. 관심 있는 물질의 CIF 파일을 `Structure.from_file("파일이름.cif")`로 읽어 이완시켜 보세요.
2. `chgnet.predict_structure(s)["m"]`으로 원자별 자기 모멘트를 볼 수 있습니다. 전이금속 산화물에서 확인해 보세요.
3. 범용 퍼텐셜의 예측을 그대로 믿으면 안 되는 경우는 언제일지 강의의 「한계와 열린 문제」 슬라이드와 연결해 생각해 보세요.

---
## 정리

| 실습에서 한 것 | 강의의 개념 |
|---|---|
| 퍼셉트론 2층으로 XOR 구현 | 은닉층과 비선형성 |
| 순전파, 역전파 손 계산과 autograd 비교 | 연쇄 법칙, 자동 미분 |
| 학습률, optimizer, batch size 실험 | 경사 하강법 |
| validation loss 감시, weight decay, dropout | 과적합과 regularization |
| 주기 경계 조건에서 이웃 찾기, Gaussian 전개 | 결정의 그래프 표현 |
| CGCNN convolution과 pooling 구현 | message passing, readout |
| 순열·회전·supercell 불변성 확인 | 대칭성 |
| 무작위 분할과 그룹 분할 비교 | 데이터 누출 |
| CHGNet으로 구조 이완 | 범용 원자간 퍼텐셜 |

Part 5와 6의 데이터는 가상의 pair potential로 만든 것입니다. 실제 연구에 적용하려면 `records`를 만드는 부분을 Materials Project 등의 구조와 DFT 물성으로 바꾸면 됩니다. CGCNN 공식 코드는 https://github.com/txie-93/cgcnn 에 있습니다.